<a href="https://colab.research.google.com/github/dylan199512/Retail-Analysis-with-Walmart-Data/blob/main/Retail_Analysis_with_Walmart_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Upload Walmart_Store_sales.csv file
from google.colab import files
uploaded = files.upload()



Saving Walmart_Store_sales.csv to Walmart_Store_sales.csv


In [ ]:
#Read file
import pandas as pd
df = pd.read_csv('/content/Walmart_Store_sales.csv')
df.head()

#Convert to datetime
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')
df.head()

# Extract useful time components (year, month, quarter) for trend analysis
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Quarter'] = df['Date'].dt.quarter

#Checking top 5 rows of dataset
df.head()


,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment,Year,Month,Quarter
0,1,2010-02-05,1643690.90,0,42.31,2.572,211.096358,8.106,2010,2,1
1,1,2010-02-12,1641957.44,1,38.51,2.548,211.242170,8.106,2010,2,1
2,1,2010-02-19,1611968.17,0,39.93,2.514,211.289143,8.106,2010,2,1
3,1,2010-02-26,1409727.59,0,46.63,2.561,211.319643,8.106,2010,2,1
4,1,2010-03-05,1554806.68,0,46.50,2.625,211.350143,8.106,2010,3,1


In [ ]:
#Identify the store with the highest total sales across the entire dataset
store_sales = df.groupby('Store')['Weekly_Sales'].sum().reset_index()
store_sales_sorted = store_sales.sort_values('Weekly_Sales', ascending=False)

max_store = store_sales_sorted.iloc[0]
print("Store with maximum sales:", int(max_store['Store']))
print("Total sales:", max_store['Weekly_Sales'])





Store with maximum sales: 20
Total sales: 301397792.46


In [ ]:
# Measure sales variability by calculating mean, standard deviation, and coefficient of variation for each store
store_stats = df.groupby('Store')['Weekly_Sales'].agg(['mean', 'std']).reset_index()
store_stats['coefficient_of_variation'] = store_stats['std'] / store_stats['mean']

store_stats_sorted = store_stats.sort_values('std', ascending=False)

max_std_store = store_stats_sorted.iloc[0]
print("Store with maximum standard deviation:", int(max_std_store['Store']))
print("Mean weekly sales:", max_std_store['mean'])
print("Standard deviation:", max_std_store['std'])
print("Coefficient of variation:", max_std_store['coefficient_of_variation'])




Store with maximum standard deviation: 14
Mean weekly sales: 2020978.4009790209
Standard deviation: 317569.9494755081
Coefficient of variation: 0.15713673600948333


In [ ]:
# Mean sales in non-holiday weeks (all stores)
mean_non_holiday = df[df['Holiday_Flag'] == 0]['Weekly_Sales'].mean()

#Keep only holiday weeks
holiday_weeks=df[df['Holiday_Flag']==1].copy()

# Map actual holiday dates (only those in your data range)

holiday_dates=[
    '2010-02-12', '2011-02-11', '2012-02-10', #Super Bowl
    '2010-09-10', '2011-09-09', '2012-09-07', #Labor Day
    '2010-11-26', '2011-11-25', '2012-11-23',  # Thanksgiving
    '2010-12-31', '2011-12-30', '2012-12-28'   # Christmas
]

holiday_weeks['Date'] = pd.to_datetime(holiday_weeks['Date'])
holiday_weeks = holiday_weeks[holiday_weeks['Date'].isin(pd.to_datetime(holiday_dates))]

# Average sales per holiday date (all stores)
holiday_sales = holiday_weeks.groupby('Date')['Weekly_Sales'].mean().reset_index()

# Add comparison column
holiday_sales['above_non_holiday_mean'] = holiday_sales['Weekly_Sales'] > mean_non_holiday

holiday_sales





,Date,Weekly_Sales,above_non_holiday_mean
0,2010-02-12,1.074148e+06,True
1,2010-09-10,1.014098e+06,False
2,2010-11-26,1.462689e+06,True
3,2010-12-31,8.985004e+05,False
4,2011-02-11,1.051915e+06,True
5,2011-09-09,1.039183e+06,False
6,2011-11-25,1.479858e+06,True
7,2011-12-30,1.023166e+06,False
8,2012-02-10,1.111320e+06,True
9,2012-09-07,1.074001e+06,True


In [ ]:
#Evaluate Q3 2012 performance by calculating growth from Q2 to Q3 for each store

# Filter only 2012 data
df_2012 = df[df['Year'] == 2012]

# Aggregate sales by Store and Quarter
quarterly_2012 = df_2012.groupby(['Store', 'Quarter'])['Weekly_Sales'].sum().reset_index()

# Pivot so quarters become columns
quarterly_pivot = quarterly_2012.pivot(index='Store', columns= 'Quarter', values='Weekly_Sales')


# Calculate growth from Q2 to Q3
quarterly_pivot['Q3_growth_from_Q2'] = (quarterly_pivot[3] - quarterly_pivot[2]) / quarterly_pivot[2]

# Sort stores in descending order of Q3 growth so the highest‑growth stores appear first
quarterly_pivot_sorted = quarterly_pivot.sort_values('Q3_growth_from_Q2', ascending=False)

quarterly_pivot_sorted.head()

Quarter,1,2,3,4,Q3_growth_from_Q2
Store,,,,,
7,7792647.21,7290859.27,8262787.39,2021262.60,0.133308
16,6400479.72,6564335.98,7121541.64,2016067.98,0.084884
35,9642858.59,10838313.00,11322421.12,3434129.81,0.044666
26,12071075.04,13155335.57,13675691.91,4074341.77,0.039555
39,18740604.09,20214128.46,20715116.23,6215814.07,0.024784


In [ ]:
#Summarize sales at the monthly and semester level to uncover seasonal patterns

monthly_sales = df.groupby(['Year', 'Month'])['Weekly_Sales'].sum().reset_index()
monthly_sales



,Year,Month,Weekly_Sales
0,2010,2,1.903330e+08
1,2010,3,1.819198e+08
2,2010,4,2.314124e+08
3,2010,5,1.867109e+08
4,2010,6,1.922462e+08
5,2010,7,2.325801e+08
6,2010,8,1.876401e+08
7,2010,9,1.772679e+08
8,2010,10,2.171618e+08
9,2010,11,2.028534e+08


In [ ]:
# Create semester column
df['Semester'] = df['Month'].apply(lambda m: 1 if m <= 6 else 2)

# Semester sales view
semester_sales = df.groupby(['Year', 'Semester'])['Weekly_Sales'].sum().reset_index()
semester_sales

,Year,Semester,Weekly_Sales
0,2010,1,9.826223e+08
1,2010,2,1.306264e+09
2,2011,1,1.127340e+09
3,2011,2,1.320860e+09
4,2012,1,1.210765e+09
5,2012,2,7.893674e+08


In [ ]:
# Filter Store 1
store_1 = df[df['Store'] == 1].copy()

# Sort by date
store_1.sort_values('Date')

# Create a day index starting at 1
store_1['Day_Index'] = (store_1 ['Date']-store_1['Date'].min()).dt.days + 1

store_1.head()

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment,Year,Month,Quarter,Semester,Day_Index
0,1,2010-02-05,1643690.90,0,42.31,2.572,211.096358,8.106,2010,2,1,1,1
1,1,2010-02-12,1641957.44,1,38.51,2.548,211.242170,8.106,2010,2,1,1,8
2,1,2010-02-19,1611968.17,0,39.93,2.514,211.289143,8.106,2010,2,1,1,15
3,1,2010-02-26,1409727.59,0,46.63,2.561,211.319643,8.106,2010,2,1,1,22
4,1,2010-03-05,1554806.68,0,46.50,2.625,211.350143,8.106,2010,3,1,1,29


In [ ]:
#Build Model A using time and economic indicators (CPI, unemployment, fuel price)

X=store_1[['Day_Index', 'CPI', 'Unemployment', 'Fuel_Price']]

# Weekly Sales

y=store_1['Weekly_Sales']

In [ ]:
#Step 3 Train/Test Data

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
#Step 4 Fit Linear Regression Model

from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)


LinearRegression()

In [ ]:
#Train/Test Split
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

# Predict on the test set
y_pred = model.predict(X_test)


# Calculate R² and RMSE
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

r2, rmse

(0.009792471764679123, np.float64(154758.98138844568))

In [ ]:
#Build a simpler baseline model using only time (Day_Index) as the predictor

X2 = store_1[['Day_Index']]
y2 = store_1['Weekly_Sales']



In [ ]:
#Train/Test Split Model B

from sklearn.model_selection import train_test_split

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42
)

In [ ]:
#Fit Model B

from sklearn.linear_model import LinearRegression

model2 = LinearRegression()
model2.fit(X2_train, y2_train)


LinearRegression()

In [ ]:
# Evaluate model performance using R² and RMSE to compare predictive accuracy

from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

# Predict on the test set
y2_pred = model2.predict(X2_test)


# Calculate R² and RMSE
r2 = r2_score(y2_test, y2_pred)
rmse = np.sqrt(mean_squared_error(y2_test, y2_pred))

r2, rmse

(-0.038902815745591734, np.float64(158518.59424817096))

In [ ]:
#Examine regression coefficients to understand how each variable influences weekly sales


coefficients = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_
})

coefficients


,Feature,Coefficient
0,Day_Index,83.170747
1,CPI,14108.215546
2,Unemployment,88317.366521
3,Fuel_Price,-86633.915496
